# TREC Robust04: Retrieval Pipeline & Experiments

**Objective**: Reproduce the three submission runs for the TREC Robust04 task.

**Approaches**:
1. **Baseline Fusion**: Combining sparse (RM3), learned sparse (SPLADE), and dense (BGE) retrieval.
2. **Structured Queries (SDM)**: Integrating Sequential Dependence Model (SDM) for phrase awareness.
3. **Neural Reranking**: Applying MonoT5 passage-level reranking for high-precision re-ordering.

## Presentation Agenda

1. **Problem Overview**: TREC Robust04 dataset and the judged/test split.
2. **Methodology**: Multi-signal fusion and neural reranking pipeline.
3. **Experiment 1 (Baseline)**: Fusing sparse, learned sparse, and dense signals.
4. **Experiment 2 (Structure)**: Improving sparse retrieval with SDM (Sequential Dependence Model).
5. **Experiment 3 (Neural)**: Passage-level reranking with MonoT5 (Final Submission).
6. **Results**: MAP performance on judged queries.

## Summary of Runs

| Run ID | Descriptive Name | Key Components |
| :--- | :--- | :--- |
| **Run 1** | **Baseline Fusion** | RM3 + SPLADE++ + SPLADE-v3 + Dense (BGE) |
| **Run 2** | **Structured Queries (SDM)** | **SDM+RM3** + SPLADE++ + Dense (BGE) |
| **Run 3** | **Neural Reranking** | Baseline Fusion + **MonoT5-3B Passage Reranking** |

## Dataset: TREC Robust04

**Why "Robust"?**
This dataset collects "hard" topics from previous TREC years—queries where systems historically failed. The goal is to improve stability and performance on difficult, ambiguous, or detail-oriented topics, rather than just optimizing for easy ones.

- **Collection**: ~528k Newswire articles (Financial Times, LA Times, FBIS, etc.) from the late 80s/early 90s.
- **Queries**: 249 total topics.
  - **Judged / Validation (50 queries)**: We have relevance labels for these. We use them to tune weights.
  - **Test / Submission (199 queries)**: We must generate rankings for these without seeing labels.

### Metric: Mean Average Precision (MAP)
MAP is the standard metric for ad-hoc retrieval. It rewards placing relevant documents at the very top of the list.

$$ \text{MAP} = \frac{1}{|Q|} \sum_{q \in Q} \text{AP}(q) $$

*Note: Because we only have 50 judged queries, we must avoid overfitting. We rely on "zero-shot" or "transfer" capabilities of modern models (SPLADE, HyDE, MonoT5) rather than training from scratch.*

## Pipeline Overview

### 1. Retrieval Components
We combine complementary retrieval signals to maximize recall and robustness:
- **Sparse (BM25 + RM3)**: Matches exact keywords. efficient but fails on vocabulary mismatch (e.g., "car" vs "automobile"). **RM3** adds pseudo-relevance feedback to expand the query with related terms from top docs.
- **Learned Sparse (SPLADE)**: Uses a BERT model to perform "sparse expansion". It learns to add relevant terms to the query/doc vector (e.g., adding "virus" to "covid"), bridging the vocabulary gap while maintaining the efficiency of inverted indexes.
- **Dense (BGE + HyDE)**: Encodes semantics into vector space. Matches queries to documents based on meaning, not just overlapping words. We enhance this with **HyDE** (Hypothetical Document Embeddings).

### 2. Score Fusion
Since each retriever $R_i$ outputs scores on a different scale (BM25 is unbounded, Cosine is [-1, 1]), we **min-max normalize** scores per query before weighted summation:

$$ S_{\text{norm}}(d) = \frac{S(d) - S_{\min}}{S_{\max} - S_{\min}} $$

$$ S_{\text{final}}(d) = \sum_{i} w_i \cdot S_{\text{norm}, i}(d) $$

### 3. Neural Reranking (Run 3)
A compute-heavy reranker is applied only to the top-$k$ candidates:
- **Passage Scoring**: **MonoT5-3B** (a T5 model fine-tuned for relevance) scores overlapping passages. It uses **Cross-Attention**, allowing deep interaction between every query token and document token.
- **Aggregation**: Max-passage score represents the document.
- **Interpolation**: The neural score is blended with the initial fusion score to preserve global retrieval signals.

## Experiment 1: Baseline Multi-Signal Fusion

**Goal**: Create a "High-Recall" candidate pool by fusing diverse strategies. No single retriever is perfect.

### Why these 4 components?
1.  **RM3 (Sparse)**: **The Safety Net**. Ensures we find documents containing the exact query words. It is robust and explainable but limited to exact matches.
2.  **SPLADE (Learned Sparse)**: **Vocabulary Expansion**. Uses BERT to predict "impact scores" for terms that *should* be in the document. It finds synonyms (e.g., query "car" matches doc "vehicle") while keeping the efficiency of sparse search.
3.  **Dense BGE (Dense)**: **Semantic Matching**. Embeds queries and docs into vector space to capture conceptual similarity, even with zero word overlap.

### Composition
We fuse these signals (Weights $W_{\text{run1}}$) to get the "best of all worlds":

$$ S_{\text{run1}} = 0.55 \cdot \text{RM3} + 0.10 \cdot \text{SPLADE}\!+\!+ + 0.15 \cdot \text{SPLADE-v3} + 0.20 \cdot \text{Dense (HyDE)} $$

## Experiment 2: Structured Queries (SDM)

**Goal**: Incorporate phrase-level evidence using the **Sequential Dependence Model (SDM)**.

### 1. Mathematical Formulation
Standard BM25 treats queries as a "bag of words" (independence assumption). SDM relaxes this by modeling dependencies between query terms using a Markov Random Field (MRF). It scores documents based on three feature functions:

Given a query $Q = (q_1, q_2, ... q_n)$ and document $D$:

$$ Score(D, Q) = \lambda_T \sum_{i} f_T(q_i, D) + \lambda_O \sum_{i} f_O(q_i, q_{i+1}, D) + \lambda_U \sum_{i} f_U(q_i, q_{i+1}, D) $$

Where:
1.  **$f_T$ (Unigrams)**: Standard keyword matching (like BM25).
    *   *Weight $\lambda_T = 0.75$*
2.  **$f_O$ (Ordered Phrases)**: Exact sequence matches (e.g., "hubble space telescope").
    *   *Constraint*: Terms must appear adjacent in order ($pos(q_{i+1}) = pos(q_i) + 1$).
    *   *Weight $\lambda_O = 0.10$*
3.  **$f_U$ (Unordered Windows)**: Terms appearing near each other (e.g., "telescope" near "hubble").
    *   *Constraint*: Terms must appear within a window of 8 words ($|pos(q_i) - pos(q_{i+1})| < 8$).
    *   *Weight $\lambda_U = 0.15$*

### 2. Implementation: SDM + RM3
In **Run 2**, we do not just use SDM alone. We combine it with **RM3 (Relevance Model 3)** for expansion.

**The Process**:
1.  **Initial Retrieval (SDM)**: We execute the SDM query (using the weights above) to retrieve the top $N$ documents.
    *   *Example Query*: `#weight(0.75 "hubble" 0.10 #1("hubble" "telescope") 0.15 #uw8("hubble" "telescope"))`
2.  **Feedback (RM3)**: We analyze the top retrieved documents to find relevant expansion terms (pseudo-relevance feedback).
3.  **Final Score**: The original SDM query is expanded with these new terms and re-executed.

### 3. Fusion Configuration
We replace the standard "Bag of Words" retrieval from Experiment 1 with this structured approach.

**Run 2 Fusion Formula**:
$$ S_{\text{run2}} = 0.60 \cdot \underbrace{\text{(SDM + RM3)}}_{\text{Structured Sparse}} + 0.25 \cdot \underbrace{\text{SPLADE}\!+\!+}_{\text{Learned Sparse}} + 0.15 \cdot \underbrace{\text{Dense}}_{\text{Semantic}} $$

This approach specifically targets queries where **word order matters** (e.g., "black bear attacks" vs "bear black"), fixing the precision errors of the baseline.

## Experiment 3: Neural Passage Reranking

**Goal**: Apply a "Cross-Encoder" (MonoT5-3B) to re-order the top candidates from the baseline.

### Concept: Cross-Attention vs. Bi-Encoders
- **Bi-Encoders (like BGE/SPLADE)**: Encode query and document *separately*. Fast (pre-computable) but shallow—they miss complex interactions (e.g., negation, subtle conditioning).
- **Cross-Encoders (MonoT5)**: Feed the query and document *together* into the transformer (`Query: q Document: d`). The model's attention mechanism can compare every query token to every document token.
  - **Pros**: Extremely high precision.
  - **Cons**: Very slow (cannot pre-compute). We only rerank the top 1000 documents.

### Addressing Long Documents (MaxP)
Robust04 documents are long newswire articles. Transformers have a 512-token limit.
**Strategy**:
1.  **Split**: Divide document into sliding windows (passages) of 1500 chars (approx 300 words).
2.  **Score**: MonoT5 scores each passage independently: $P(\text{relevant} \mid q, p)$.
3.  **Aggregate (MaxP)**: The score of a document is the score of its *best* passage. If one paragraph answers the query, the whole document is relevant.

$$ S_{\text{final}} = \alpha \cdot \text{norm}(S_{\text{baseline}}) + (1-\alpha) \cdot \text{norm}(S_{\text{MonoT5}}) $$

Parameters: Top-$N = 1000$, $\alpha = 0.3$. This interpolation keeps the global signal from the baseline while boosting exact-match precision.

In [14]:
import os
import re
import json
import math
import time
import gzip
import pickle
import hashlib
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

# Prevent Lucene memory-segment issues in some environments
os.environ.setdefault(
    "JAVA_TOOL_OPTIONS",
    "-Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false",
)

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
)

from pyserini.pyclass import autoclass

# IMPORTANT: importing SpladeQueryEncoder from pyserini.encode (public) can trigger optional faiss modules.
# Using the private module avoids that optional import path.
from pyserini.encode._splade import SpladeQueryEncoder
from pyserini.search.lucene import LuceneSearcher, LuceneImpactSearcher, LuceneHnswDenseSearcher

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Runtime Device: {DEVICE.upper()} (Torch {torch.__version__})")

Runtime Device: CUDA (Torch 2.9.0+cu128)


# Setup & Configuration

In [ ]:
# Configuration & Hyperparameters

PROJECT_ROOT = Path('.').resolve()
QUERIES_PATH = PROJECT_ROOT / 'Files-20260104' / 'queriesROBUST.txt'
QRELS_JUDGED_PATH = PROJECT_ROOT / 'Files-20260104' / 'qrels_50_Queries'
HYDE_JSONL_PATH = PROJECT_ROOT / 'hyde_all_hypothetical_docs.jsonl'
CACHE_DIR = Path('/workspace/.cache')

# Output Files
OUT_RUN_1 = PROJECT_ROOT / 'run_1.res'
OUT_RUN_2 = PROJECT_ROOT / 'run_2.res'
OUT_RUN_3 = PROJECT_ROOT / 'run_3.res'

# Retrieval Settings
QID_SET = 'test'  # 'judged' (50) | 'test' (199)
K = 1000

# 1. Sparse Retrieval (RM3)
RM3_INDEX = 'robust04'
BM25_K1 = 0.9
BM25_B = 0.4
RM3_FB_TERMS = 20
RM3_FB_DOCS = 5
RM3_OQW = 0.5

# 2. Structured Queries (SDM)
RUN2_USE_SDM_RM3 = True
SDM_TERM_WEIGHT = 0.75
SDM_ORDERED_WINDOW_WEIGHT = 0.10
SDM_UNORDERED_WINDOW_WEIGHT = 0.15

# 3. Learned Sparse & Dense Retrieval
SPLADEPP_INDEX = 'beir-v1.0.0-robust04.splade-pp-ed'
SPLADEPP_MODEL = 'naver/splade-cocondenser-ensembledistil'
SPLADEV3_INDEX = 'beir-v1.0.0-robust04.splade-v3'
SPLADEV3_MODEL = 'naver/splade-v3-distilbert'
DENSE_INDEX = 'beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw'
DENSE_ENCODER = 'BgeBaseEn15'
DENSE_EF_SEARCH = 1000

# Fusion Weights
W_RUN2 = [0.60, 0.25, 0.15]           # SDM+RM3, SPLADE++, Dense
W_RUN3 = [0.55, 0.10, 0.15, 0.20]     # RM3, SPLADE++, SPLADE-v3, Dense

# Query Sources (Orig vs HyDE)
QUERY_SOURCE_RM3 = 'orig'
QUERY_SOURCE_SPLADEPP = 'orig'
QUERY_SOURCE_SPLADEV3 = 'orig'
QUERY_SOURCE_DENSE = 'orig_hyde'

# 4. Neural Reranking (MonoT5)
RERANK3_MONOT5_PASSAGES = True
MONOT5P_MODEL = 'cramraj8/duqgen-monot5-3b-robust04-1k'
MONOT5P_TOP_N = 1000
MONOT5P_ALPHA = 0.3
MONOT5P_BATCH_SIZE = 16
MONOT5P_MAX_LENGTH = 512
MONOT5P_DOC_MAX_CHARS = 20000
MONOT5P_PASSAGE_CHARS = 1500
MONOT5P_STRIDE_CHARS = 1200
MONOT5P_MAX_PASSAGES = 15
MONOT5P_SCORE_TOP_N = 1000
MONOT5P_SCORE_MAX_PASSAGES = 15
MONOT5P_AGG = 'max'
MONOT5P_AVG_TOPK = 3
MONOT5P_SOFTMAX_TEMP = 1.0
MONOT5P_HYBRID_LAMBDA = 0.5
MONOT5P_FP16 = True

# Execution Flags
FORCE_REGEN_HYDE = False
FORCE_REGEN_RUNS = True
FORCE_CACHE_REFRESH = False

print(f"Configured for: {QID_SET.upper()} split")
print(f"Outputs: {OUT_RUN_1.name}, {OUT_RUN_2.name}, {OUT_RUN_3.name}")

Configured for: TEST split
Outputs: run_1.res, run_2.res, run_3.res


### Utility: Disk Caching

In [16]:
def _update_hash(h: "hashlib._Hash", obj: Any) -> None:
    if obj is None:
        h.update(b"n")
        return
    if isinstance(obj, bool):
        h.update(b"b1" if obj else b"b0")
        return
    if isinstance(obj, int):
        h.update(b"i")
        h.update(str(obj).encode("utf-8"))
        h.update(b";")
        return
    if isinstance(obj, float):
        h.update(b"f")
        h.update(repr(obj).encode("utf-8"))
        h.update(b";")
        return
    if isinstance(obj, str):
        h.update(b"s")
        h.update(obj.encode("utf-8"))
        h.update(b";")
        return
    if isinstance(obj, bytes):
        h.update(b"y")
        h.update(obj)
        h.update(b";")
        return
    if isinstance(obj, (list, tuple)):
        h.update(b"[")
        for x in obj:
            _update_hash(h, x)
            h.update(b",")
        h.update(b"]")
        return
    if isinstance(obj, dict):
        h.update(b"{")
        for k in sorted(obj.keys(), key=lambda x: (str(type(x)), repr(x))):
            _update_hash(h, k)
            h.update(b":")
            _update_hash(h, obj[k])
            h.update(b",")
        h.update(b"}")
        return

    h.update(b"r")
    h.update(repr(obj).encode("utf-8"))
    h.update(b";")


def make_hash(obj: Any) -> str:
    h = hashlib.sha256()
    _update_hash(h, obj)
    return h.hexdigest()


@dataclass
class DiskCache:
    cache_dir: Path
    enabled: bool = True
    refresh: bool = False

    def _path(self, namespace: str, key: str) -> Path:
        subdir = self.cache_dir / namespace / key[:2] / key[2:4]
        return subdir / f"{key}.pkl.gz"

    def get(self, namespace: str, key_obj: Any) -> Optional[Any]:
        if (not self.enabled) or self.refresh:
            return None
        key = make_hash(key_obj)
        path = self._path(namespace, key)
        if not path.exists():
            return None
        try:
            with gzip.open(path, "rb") as f:
                return pickle.load(f)
        except Exception:
            return None

    def set(self, namespace: str, key_obj: Any, value: Any) -> None:
        if not self.enabled:
            return
        key = make_hash(key_obj)
        path = self._path(namespace, key)
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + ".tmp")
        with gzip.open(tmp, "wb") as f:
            pickle.dump(value, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, path)


disk_cache = DiskCache(cache_dir=CACHE_DIR, enabled=True, refresh=bool(FORCE_CACHE_REFRESH))

## 1. Data Loading (Queries & Splits)

In [17]:
def read_queries_tsv(path: Path) -> Dict[str, str]:
    queries: Dict[str, str] = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            qid, query = line.split('\t', 1)
            queries[str(qid)] = str(query)
    return queries


all_queries = read_queries_tsv(QUERIES_PATH)
all_qids = list(all_queries.keys())
train_qids = all_qids[:50]
test_qids = all_qids[50:]

if QID_SET == 'judged':
    target_qids = train_qids
elif QID_SET == 'all':
    target_qids = all_qids
else:
    target_qids = test_qids

target_queries = {qid: all_queries[qid] for qid in target_qids}

print('total_queries:', len(all_queries))
print('train_qids:', len(train_qids))
print('test_qids:', len(test_qids))
print('target_qids:', len(target_qids))
print('target_qid_first_last:', target_qids[0], target_qids[-1])


total_queries: 249
train_qids: 50
test_qids: 199
target_qids: 199
target_qid_first_last: 351 700


## 2. HyDE Generation (Hypothetical Document Embeddings)

**Concept**: Dense retrievers often fail because queries are short/keyword-based ("gun control") while documents are long/prose-based. They map to different parts of the embedding space.
**HyDE (Hypothetical Document Embeddings)** bridges this gap using an LLM (Zephyr-7B-beta) to hallucinate a "fake" relevant document.

### The Process
1.  **Prompt**: "Write a news passage about [Query]."
2.  **Generate**: The LLM creates a hypothetical document.
3.  **Encode**: We embed this hypothetical document instead of the original query.
4.  **Retrieve**: We search for real documents semantically similar to the hallucination.

### Qualitative Analysis: Why it works (and fails)

We analyzed specific generations to understand the mechanics:

#### The Win: QID 303 "Hubble Telescope Achievements"
*   **Original Query**: Short, abstract. Matches only documents explicitly saying "achievements".
*   **HyDE Generation**:
    > "The Hubble Space Telescope... capturing the **first-ever image of a distant supernova**, revealing the **age of the universe**... **dark energy**... **exoplanets**..."
*   **Result**: MAP **0.0916 → 0.3299**. The LLM performed **Latent Term Expansion**, adding specific achievement terms that appear in the corpus.

#### The Loss: QID 310 "Radio Waves Brain Cancer"
*   **Original Query**: Ambiguous. Likely looking for health risks (Does X cause Y?).
*   **HyDE Generation**:
    > "Scientists... discovered a new way to **destroy brain cancer cells using radio waves**... **radiofrequency ablation**..."
*   **Result**: MAP **0.3554 → 0.0287**.
*   **Failure Mode**: **Semantic Drift**. The LLM interpreted the query as "How to use radio waves to treat cancer" instead of "Do radio waves cause cancer". The retrieval vector shifted to a completely different topic (Therapy vs Risk).

### Fusion Impact
Despite these failures, mixing HyDE with the original query (Fusion) yields a net positive (**MAP 0.2972 → 0.3002**).

---

### Beyond HyDE: Advanced Query Transformations
Our exploration of the `RAG_Techniques` repository highlights other methods to manipulate the query vector space:

1.  **Query Rewriting**:
    *   *Concept*: Use an LLM to rewrite the query to be more specific/detailed before retrieval.
    *   *Example*: "Climate change impacts" $\rightarrow$ "Specific effects on biodiversity, sea levels..."
2.  **Step-back Prompting**:
    *   *Concept*: Generate a broader, more abstract query to retrieve high-level context.
    *   *Example*: "Impacts of X on Y" $\rightarrow$ "What is the history of X?"
3.  **Sub-query Decomposition**:
    *   *Concept*: Break complex queries into atomic sub-questions, retrieve for each, and synthesize.
    *   *Use Case*: Multi-faceted queries.

### Inverse Approach: HyPE (Hypothetical Prompt Embeddings)
Instead of transforming the *query*, we can transform the *document* index.
*   **Method**: For every document chunk, use an LLM to generate "Hypothetical Questions" that this chunk answers.
*   **Index**: Index the *questions* instead of the *content*.
*   **Benefit**: Matches user questions directly to hypothetical questions (Question-to-Question matching), avoiding the Query-to-Doc modality gap.

### Optimization Note
HyDE's failure on QID 310 suggests **Prompt Engineering** is critical.
*   *Current*: "Write a news passage..."
*   *Better*: "Write a passage discussing the controversy/risks/impact of..."
By tuning the instruction, we can steer the hallucination away from "Treatment" and towards "Risk", fixing the drift.

In [ ]:
def _clean_hyp_text(text: str) -> str:
    s = (text or "").strip()
    if not s:
        return ""
    low = s.lower()
    if "explanation:" in low:
        idx = low.index("explanation:")
        s = s[:idx].strip()
    for prefix in ["title:", "passage:"]:
        if s.lower().startswith(prefix):
            s = s[len(prefix):].strip()
    return s


def load_qid_text_jsonl(path: Path) -> Dict[str, str]:
    out: Dict[str, str] = {}
    if not path.exists():
        return out
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            qid = str(rec.get('qid', '')).strip()
            txt = _clean_hyp_text(str(rec.get('text', '')))
            if qid and txt:
                out[qid] = txt
    return out


def _ensure_hyde_jsonl_exists(
    *,
    queries: Dict[str, str],
    out_path: Path,
    force_regen: bool,
) -> Dict[str, str]:
    # Load existing (if any)
    existing = {} if force_regen else load_qid_text_jsonl(out_path)
    existing_qids = set(existing.keys())

    missing = [(qid, queries[qid]) for qid in queries.keys() if qid not in existing_qids]

    if (not missing) and (len(existing) == len(queries)):
        print(f"HyDE JSONL OK: {out_path} has {len(existing)}/{len(queries)} qids")
        return existing

    if force_regen and out_path.exists():
        # Rebuild file from scratch
        out_path.unlink()
        existing = {}
        existing_qids = set()
        missing = [(qid, queries[qid]) for qid in queries.keys()]

    print(f"HyDE JSONL needs generation: have {len(existing_qids)}/{len(queries)}; missing {len(missing)}")

    # Fixed seed for best-effort reproducibility
    torch.manual_seed(0)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(0)

    print('Loading HyDE model:', HYDE_MODEL_NAME)
    tok = AutoTokenizer.from_pretrained(HYDE_MODEL_NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    # Model load (optionally 4-bit)
    if HYDE_LOAD_IN_4BIT:
        try:
            from transformers import BitsAndBytesConfig

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16,
            )
            model = AutoModelForCausalLM.from_pretrained(
                HYDE_MODEL_NAME,
                quantization_config=bnb_config,
                device_map='auto',
            )
        except Exception as e:
            print('4-bit load failed, falling back to float16:', repr(e))
            model = AutoModelForCausalLM.from_pretrained(
                HYDE_MODEL_NAME,
                torch_dtype=torch.float16,
                device_map='auto',
            )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            HYDE_MODEL_NAME,
            torch_dtype=torch.float16,
            device_map='auto',
        )

    model.eval()

    bs = max(1, int(HYDE_BATCH_SIZE))
    gen_kwargs = {
        'max_new_tokens': int(HYDE_MAX_NEW_TOKENS),
        'do_sample': (not HYDE_GREEDY),
        'pad_token_id': int(tok.pad_token_id),
    }
    if not HYDE_GREEDY:
        gen_kwargs.update({'temperature': float(HYDE_TEMPERATURE), 'top_p': float(HYDE_TOP_P)})

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Append mode allows resume; if we force-regenerated we already deleted the file.
    with out_path.open('a', encoding='utf-8') as f_out:
        for start in range(0, len(missing), bs):
            batch = missing[start:start+bs]
            qids = [x[0] for x in batch]

            batch_prompts = []
            for _, q in batch:
                batch_prompts.append(
                    [
                        {
                            'role': 'system',
                            'content': 'You are a helpful assistant. Write a short news passage that answers the given query.',
                        },
                        {'role': 'user', 'content': f"Query: {q}\nPassage:"},
                    ]
                )

            prompt_texts = [tok.apply_chat_template(p, tokenize=False, add_generation_prompt=True) for p in batch_prompts]
            enc = tok(prompt_texts, return_tensors='pt', padding=True)
            input_len = int(enc['input_ids'].shape[1])
            enc = enc.to(model.device)

            with torch.inference_mode():
                outputs = model.generate(**enc, **gen_kwargs)

            for i, qid in enumerate(qids):
                gen_ids = outputs[i][input_len:].tolist()
                gen_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
                rec = {'qid': str(qid), 'text': gen_text}
                f_out.write(json.dumps(rec) + '\n')
                existing[str(qid)] = _clean_hyp_text(gen_text)

            f_out.flush()

            if (start // bs + 1) % 10 == 0:
                done = min(start + bs, len(missing))
                print(f"HyDE generated {done}/{len(missing)} missing qids")

    # Final load ensures we return a normalized mapping even if generation produced empty strings.
    hyde_docs = load_qid_text_jsonl(out_path)
    if len(hyde_docs) != len(queries):
        missing2 = sorted(set(queries.keys()) - set(hyde_docs.keys()))
        raise RuntimeError(f"HyDE JSONL incomplete after generation: missing {len(missing2)} qids (e.g. {missing2[:10]})")

    print(f"HyDE JSONL ready: {out_path} has {len(hyde_docs)}/{len(queries)} qids")
    return hyde_docs


# Generate/load HyDE for *all* queries, because query sources can be applied across qid sets.
hyde_docs_all = _ensure_hyde_jsonl_exists(
    queries=all_queries,
    out_path=HYDE_JSONL_PATH,
    force_regen=bool(FORCE_REGEN_HYDE),
)


HyDE JSONL OK: /root/textretfinal/hyde_all_hypothetical_docs.jsonl has 249/249 qids


### Utility: Query Source Resolution
*Handles switching between Original and HyDE query text.*

In [19]:
def resolve_query_source(
    *,
    qid: str,
    orig_query: str,
    source: str,
    hyde_docs: Dict[str, str],
) -> str:
    src = str(source)
    if src == 'orig':
        return orig_query
    if src == 'hyde':
        return hyde_docs.get(qid, orig_query)
    if src == 'orig_hyde':
        hyp = hyde_docs.get(qid, '')
        return orig_query if not hyp else (orig_query + ' ' + hyp)
    return orig_query


# quick smoke-check
qid0 = target_qids[0]
print('example qid:', qid0)
print('orig:', all_queries[qid0])
print('dense_query_source:', QUERY_SOURCE_DENSE)
print('dense_query_text_prefix:', resolve_query_source(qid=qid0, orig_query=all_queries[qid0], source=QUERY_SOURCE_DENSE, hyde_docs=hyde_docs_all)[:120])


example qid: 351
orig: falkland petroleum exploration
dense_query_source: orig_hyde
dense_query_text_prefix: falkland petroleum exploration The Falkland Islands government has announced the award of four new licences for oil and 


## 3. Retrieval & Fusion Functions

Implements min-max normalization and weighted summation.

In [20]:
@dataclass
class SearchArtifacts:
    docids_scores: Dict[str, float]
    ranked: List[Tuple[str, float]]


def retrieve(searcher, query: str, k: int, query_generator=None) -> SearchArtifacts:
    if query_generator is None:
        hits = searcher.search(query, k=k)
    else:
        hits = searcher.search(query, k=k, query_generator=query_generator)
    ranked = [(h.docid, float(h.score)) for h in hits]
    scores = {docid: score for docid, score in ranked}
    return SearchArtifacts(docids_scores=scores, ranked=ranked)


def minmax_norm(scores_dict: Dict[str, float]) -> Dict[str, float]:
    if not scores_dict:
        return {}
    vals = list(scores_dict.values())
    mn, mx = min(vals), max(vals)
    if mx - mn < 1e-9:
        return {d: 0.0 for d in scores_dict}
    return {d: (float(s) - float(mn)) / (float(mx) - float(mn)) for d, s in scores_dict.items()}


def fuse_weighted_minmax(
    runs_scores: List[Dict[str, float]],
    weights: List[float],
    depth: int,
) -> List[Tuple[str, float]]:
    norms = [minmax_norm(rs) for rs in runs_scores]
    docs = set()
    for n in norms:
        docs |= set(n.keys())

    fused_scores: Dict[str, float] = {}
    for d in docs:
        s = 0.0
        for w, n in zip(weights, norms):
            s += float(w) * float(n.get(d, 0.0))
        fused_scores[d] = float(s)

    ranked = sorted(fused_scores.items(), key=lambda x: (-x[1], x[0]))
    return ranked[: int(depth)]


def ensure_k(
    ranked: List[Tuple[str, float]],
    fallback: List[Tuple[str, float]],
    k: int,
) -> List[Tuple[str, float]]:
    if len(ranked) >= int(k):
        return ranked[: int(k)]

    seen = {d for d, _ in ranked}
    out = list(ranked)
    for d, s in fallback:
        if d in seen:
            continue
        out.append((d, float(s)))
        seen.add(d)
        if len(out) >= int(k):
            break
    return out


def write_trec_run(path: Path, run: Dict[str, List[Tuple[str, float]]], tag: str) -> None:
    with path.open('w', encoding='utf-8') as f:
        for qid in sorted(run.keys(), key=int):
            ranked = run[qid]
            for rank, (docid, score) in enumerate(ranked, start=1):
                f.write(f"{qid} Q0 {docid} {rank} {float(score):.6f} {tag}\n")


def chunked(items: List[str], batch_size: int) -> Iterable[List[str]]:
    bs = int(batch_size)
    if bs <= 0:
        raise ValueError('batch_size must be > 0')
    for i in range(0, len(items), bs):
        yield items[i:i+bs]


### Utility: Fetching Raw Document Text
*Retrieves full document content from the Lucene index for reranking.*

In [21]:
_TAG_RE = re.compile(r"<[^>]+>")
_WS_RE = re.compile(r"\s+")


def raw_to_text(raw: str) -> str:
    s = _TAG_RE.sub(" ", raw or "")
    s = _WS_RE.sub(" ", s)
    return s.strip()


def fetch_doc_texts_disk_cached(
    searcher: LuceneSearcher,
    docids: List[str],
    mem_cache: Dict[str, str],
    max_chars: int,
    disk_cache: DiskCache,
) -> List[str]:
    out: List[str] = []
    for docid in docids:
        if docid in mem_cache:
            out.append(mem_cache[docid])
            continue

        key = {"index": RM3_INDEX, "docid": str(docid), "max_chars": int(max_chars)}
        txt = disk_cache.get("doc_texts", key)
        if txt is None:
            try:
                doc = searcher.doc(str(docid))
                raw = "" if doc is None else (doc.raw() or "")
            except Exception:
                raw = ""
            txt = raw_to_text(raw)
            if int(max_chars) > 0:
                txt = txt[: int(max_chars)]
            disk_cache.set("doc_texts", key, txt)

        mem_cache[str(docid)] = str(txt)
        out.append(str(txt))

    return out


## 4. MonoT5 Scoring Functions

Logic for splitting documents into passages, scoring with MonoT5, and aggregating via MaxP.

In [22]:
def _split_passages(text: str, passage_chars: int, stride_chars: int, max_passages: int) -> List[str]:
    t = text or ""
    if int(passage_chars) <= 0:
        return [t]
    stride = int(stride_chars)
    if stride <= 0:
        stride = int(passage_chars)
    mp = int(max_passages)
    if mp <= 0:
        mp = 1

    out: List[str] = []
    i = 0
    while i < len(t) and len(out) < mp:
        seg = t[i:i+int(passage_chars)]
        if seg:
            out.append(seg)
        i += stride
    if not out:
        out = [""]
    return out


def compute_monot5_passage_raw_scores(
    tokenizer: AutoTokenizer,
    model: AutoModelForSeq2SeqLM,
    true_id: int,
    false_id: int,
    query: str,
    docids: List[str],
    doc_texts: List[str],
    device: str,
    batch_size: int,
    max_length: int,
    passage_chars: int,
    stride_chars: int,
    max_passages: int,
) -> Dict[str, List[float]]:
    decoder_start = model.config.decoder_start_token_id
    if decoder_start is None:
        decoder_start = tokenizer.pad_token_id

    ex_docids: List[str] = []
    ex_texts: List[str] = []
    for d, t in zip(docids, doc_texts):
        passages = _split_passages(
            t,
            passage_chars=int(passage_chars),
            stride_chars=int(stride_chars),
            max_passages=int(max_passages),
        )
        for p in passages:
            ex_docids.append(str(d))
            ex_texts.append(f"Query: {query} Document: {p} Relevant:")

    doc_to_scores: Dict[str, List[float]] = defaultdict(list)

    with torch.no_grad():
        for batch_docids, batch_text in zip(chunked(ex_docids, int(batch_size)), chunked(ex_texts, int(batch_size))):
            enc = tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            decoder_input_ids = torch.full(
                (len(batch_text), 1),
                int(decoder_start),
                dtype=torch.long,
                device=device,
            )
            logits = model(**enc, decoder_input_ids=decoder_input_ids).logits
            step = logits[:, 0, :]
            batch_scores = (step[:, true_id] - step[:, false_id]).detach().cpu().tolist()
            for d, s in zip(batch_docids, batch_scores):
                doc_to_scores[str(d)].append(float(s))

    return {str(d): (doc_to_scores.get(str(d), []) or []) for d in docids}


def aggregate_monot5_passage_scores(
    raw_scores: Dict[str, List[float]],
    docids: List[str],
    agg: str,
    avg_topk: int,
    max_passages: int,
    softmax_temp: float,
    hybrid_lambda: float,
) -> Dict[str, float]:
    k_passages = max(1, int(max_passages))
    out: Dict[str, float] = {}
    for d in docids:
        scores = (raw_scores.get(str(d), []) or [])[:k_passages]
        if not scores:
            out[str(d)] = 0.0
            continue

        if str(agg) == 'avg_topk':
            k_take = max(1, int(avg_topk))
            topk = sorted(scores, reverse=True)[:k_take]
            out[str(d)] = float(sum(topk) / float(len(topk)))
        elif str(agg) == 'softmax':
            t = float(softmax_temp)
            if not (t > 0.0):
                t = 1.0
            m = float(max(scores)) / t
            exps = [math.exp(float(s) / t - m) for s in scores]
            denom = float(sum(exps))
            if denom <= 0.0:
                out[str(d)] = float(max(scores))
            else:
                out[str(d)] = float(sum(e * float(s) for e, s in zip(exps, scores)) / denom)
        elif str(agg) == 'hybrid':
            lam = float(hybrid_lambda)
            if lam < 0.0:
                lam = 0.0
            if lam > 1.0:
                lam = 1.0
            max_s = float(max(scores))
            k_take = max(1, int(avg_topk))
            topk = sorted(scores, reverse=True)[:k_take]
            avg_s = float(sum(topk) / float(len(topk)))
            out[str(d)] = float(lam * max_s + (1.0 - lam) * avg_s)
        else:
            out[str(d)] = float(max(scores))

    return out


## 5. Initialize Retrievers
*RM3, SPLADE++, SPLADE-v3, Dense BGE*

In [23]:
def init_searchers(device: str):
    rm3 = LuceneSearcher.from_prebuilt_index(RM3_INDEX)
    rm3.set_bm25(float(BM25_K1), float(BM25_B))
    rm3.set_rm3(int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW))

    bm25 = LuceneSearcher.from_prebuilt_index(RM3_INDEX)
    bm25.set_bm25(float(BM25_K1), float(BM25_B))

    spladepp_encoder = SpladeQueryEncoder(SPLADEPP_MODEL, device=device)
    spladepp = LuceneImpactSearcher.from_prebuilt_index(SPLADEPP_INDEX, spladepp_encoder)

    spladev3_encoder = SpladeQueryEncoder(SPLADEV3_MODEL, device=device)
    spladev3 = LuceneImpactSearcher.from_prebuilt_index(SPLADEV3_INDEX, spladev3_encoder)

    dense = LuceneHnswDenseSearcher.from_prebuilt_index(
        DENSE_INDEX,
        ef_search=int(DENSE_EF_SEARCH),
        encoder=DENSE_ENCODER,
    )

    return rm3, bm25, spladepp, spladev3, dense


t0 = time.time()
rm3, bm25, spladepp, spladev3, dense = init_searchers(DEVICE)
print(f"Initialized 4 retrievers in {round(time.time() - t0, 1)} sec")

SdmQueryGenerator = autoclass('io.anserini.search.query.SdmQueryGenerator')
sdm_query_generator = SdmQueryGenerator(
    float(SDM_TERM_WEIGHT),
    float(SDM_ORDERED_WINDOW_WEIGHT),
    float(SDM_UNORDERED_WINDOW_WEIGHT),
)
# SDM generator ready for Run 2

KeyboardInterrupt: 

### Initialize Reranker (MonoT5)
*Loads the 3B parameter seq2seq model (if enabled).*

## 6. Execution: Generating Runs and Evaluating MAP

The following blocks generate the three runs and compute MAP on the **50 judged queries**.

In [ ]:
monot5p_tokenizer = None
monot5p_model = None
true_id = None
false_id = None

if bool(RERANK3_MONOT5_PASSAGES):
    t0 = time.time()
    print(f"Loading Reranker: {MONOT5P_MODEL}...")
    monot5p_tokenizer = AutoTokenizer.from_pretrained(MONOT5P_MODEL)

    load_kwargs = {}
    if bool(MONOT5P_FP16) and str(DEVICE).startswith('cuda'):
        load_kwargs['torch_dtype'] = torch.float16

    monot5p_model = AutoModelForSeq2SeqLM.from_pretrained(MONOT5P_MODEL, **load_kwargs)
    monot5p_model.to(DEVICE)
    if bool(MONOT5P_FP16) and str(DEVICE).startswith('cuda'):
        monot5p_model.half()
    monot5p_model.eval()

    true_ids = monot5p_tokenizer.encode('true', add_special_tokens=False)
    false_ids = monot5p_tokenizer.encode('false', add_special_tokens=False)
    if (not true_ids) or (not false_ids):
        raise RuntimeError("could not tokenize 'true'/'false'")
    true_id = int(true_ids[0])
    false_id = int(false_ids[0])

    print(f"Reranker initialized in {round(time.time() - t0, 1)} sec")

# in-notebook doc text memoization
_doc_text_cache: Dict[str, str] = {}

Loading Reranker: cramraj8/duqgen-monot5-3b-robust04-1k...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.06it/s]


Reranker initialized in 4.2 sec


### Run Generation Logic

We generate the three submissions:
1. **Baseline Fusion**: Weighted combination of RM3, SPLADE++, SPLADE-v3, and Dense.
2. **Structured Queries (SDM)**: Replacing RM3 with SDM+RM3 in the fusion.
3. **Neural Reranking**: Re-scoring the top-1000 Baseline Fusion candidates with MonoT5.

# Evaluation Helpers (MAP Computation)

In [ ]:
def read_qrels(path: Path) -> Dict[str, Dict[str, int]]:
    qrels: Dict[str, Dict[str, int]] = defaultdict(dict)
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 4:
                continue
            qid, _, docid, rel_s = parts[:4]
            try:
                rel = int(rel_s)
            except Exception:
                continue
            qrels[str(qid)][str(docid)] = int(rel)
    return dict(qrels)


def average_precision(ranked_docids: List[str], qrel: Dict[str, int]) -> float:
    rel_total = sum(1 for _, r in qrel.items() if int(r) > 0)
    if rel_total <= 0:
        return 0.0

    hits = 0
    s = 0.0
    for i, d in enumerate(ranked_docids, start=1):
        if int(qrel.get(str(d), 0)) > 0:
            hits += 1
            s += float(hits) / float(i)
    return float(s) / float(rel_total)


def mean_ap(run: Dict[str, List[Tuple[str, float]]], qrels: Dict[str, Dict[str, int]]) -> float:
    if not qrels:
        return 0.0
    aps: List[float] = []
    for qid in sorted(qrels.keys(), key=int):
        ranked = run.get(str(qid), [])
        ranked_docids = [d for d, _ in ranked]
        aps.append(average_precision(ranked_docids, qrels[str(qid)]))
    return float(sum(aps) / float(len(aps)))


def evaluate_judged_maps() -> Dict[str, float]:
    if 'generate_all_runs' not in globals():
        raise RuntimeError('generate_all_runs is not defined yet; run the run-generation section first')

    qrels_judged = read_qrels(QRELS_JUDGED_PATH)
    judged_qids = sorted(list(qrels_judged.keys()), key=int)

    _prev_target_qids = target_qids
    try:
        globals()['target_qids'] = [str(q) for q in judged_qids]
        t0 = time.time()
        judged_run_1, judged_run_2, judged_run_3 = generate_all_runs()
        print('judged run generation done in', round(time.time() - t0, 1), 'sec')
    finally:
        globals()['target_qids'] = _prev_target_qids

    return {
        'run_1': mean_ap(judged_run_1, qrels_judged),
        'run_2': mean_ap(judged_run_2, qrels_judged),
        'run_3': mean_ap(judged_run_3, qrels_judged),
    }


## Interpretation of Results

**What to expect:**
1.  **Baseline (Run 1)**: Should provide a strong foundation (typically MAP ~0.29-0.30 on Robust04). The combination of dense and sparse retrieval ensures high recall.
2.  **Structured Queries (Run 2)**: Adding SDM typically improves precision for queries with phrases (e.g., "Hubble Telescope Achievements"). We expect a modest boost over the baseline (MAP +0.01-0.02).
3.  **Neural Reranking (Run 3)**: This is the biggest jump. MonoT5, being a cross-encoder, can "read" the documents and understand nuance that bi-encoders miss. We expect a significant improvement (MAP ~0.35+).

**Key Takeaway**: While efficient retrieval (SPLADE/Dense) gets us *candidates*, deep interaction models (MonoT5) are essential for *precision* at the top of the ranking.

In [ ]:
def generate_all_runs() -> Tuple[
    Dict[str, List[Tuple[str, float]]],
    Dict[str, List[Tuple[str, float]]],
    Dict[str, List[Tuple[str, float]]],
]:
    run_1: Dict[str, List[Tuple[str, float]]] = {}
    run_2: Dict[str, List[Tuple[str, float]]] = {}
    run_3: Dict[str, List[Tuple[str, float]]] = {}

    for i, qid in enumerate(target_qids, start=1):
        query = all_queries[qid]

        q_rm3 = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_RM3),
            hyde_docs=hyde_docs_all,
        )
        q_pp = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_SPLADEPP),
            hyde_docs=hyde_docs_all,
        )
        q_v3 = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_SPLADEV3),
            hyde_docs=hyde_docs_all,
        )
        q_dense = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_DENSE),
            hyde_docs=hyde_docs_all,
        )

        baseline_key = {
            "qid": str(qid),
            "query": str(query),
            "query_sources": {
                "rm3": str(QUERY_SOURCE_RM3),
                "spladepp": str(QUERY_SOURCE_SPLADEPP),
                "spladev3": str(QUERY_SOURCE_SPLADEV3),
                "dense": str(QUERY_SOURCE_DENSE),
            },
            "query_texts": {
                "rm3": str(q_rm3),
                "spladepp": str(q_pp),
                "spladev3": str(q_v3),
                "dense": str(q_dense),
            },
            "k": int(K),
            "rm3": {
                "index": str(RM3_INDEX),
                "bm25": [float(BM25_K1), float(BM25_B)],
                "rm3": [int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW)],
            },
            "run2_sparse": {
                "use_sdm_rm3": bool(RUN2_USE_SDM_RM3),
                "sdm": [float(SDM_TERM_WEIGHT), float(SDM_ORDERED_WINDOW_WEIGHT), float(SDM_UNORDERED_WINDOW_WEIGHT)],
            },
            "spladepp_index": str(SPLADEPP_INDEX),
            "spladev3_index": str(SPLADEV3_INDEX),
            "dense_index": str(DENSE_INDEX),
            "dense_ef_search": int(DENSE_EF_SEARCH),
            "w_run2": list(W_RUN2),
            "w_run3": list(W_RUN3),
        }

        cached_baseline = disk_cache.get("generate_runs_baseline", baseline_key)
        if cached_baseline is None:
            rm3_art = retrieve(rm3, q_rm3, k=int(K))
            pp_art = retrieve(spladepp, q_pp, k=int(K))
            v3_art = retrieve(spladev3, q_v3, k=int(K))
            dense_art = retrieve(dense, q_dense, k=int(K))

            if bool(RUN2_USE_SDM_RM3):
                run2_sparse_art = retrieve(rm3, q_rm3, k=int(K), query_generator=sdm_query_generator)
            else:
                run2_sparse_art = rm3_art

            fallback_zero = [(d, 0.0) for d, _ in rm3_art.ranked]

            fused2 = fuse_weighted_minmax(
                [run2_sparse_art.docids_scores, pp_art.docids_scores, dense_art.docids_scores],
                list(W_RUN2),
                depth=int(K),
            )
            fused2 = ensure_k(fused2, fallback_zero, k=int(K))

            fused3 = fuse_weighted_minmax(
                [rm3_art.docids_scores, pp_art.docids_scores, v3_art.docids_scores, dense_art.docids_scores],
                list(W_RUN3),
                depth=int(K),
            )
            fused3 = ensure_k(fused3, fallback_zero, k=int(K))

            disk_cache.set("generate_runs_baseline", baseline_key, (rm3_art.ranked[: int(K)], fused2, fused3))
        else:
            rm3_ranked, fused2, fused3 = cached_baseline
            fallback_zero = [(d, 0.0) for d, _ in rm3_ranked]
            fused2 = ensure_k(list(fused2), fallback_zero, k=int(K))
            fused3 = ensure_k(list(fused3), fallback_zero, k=int(K))

        fused3_base = list(fused3)
        run_1[str(qid)] = fused3_base
        run_2[str(qid)] = list(fused2)

        fused3_for_rerank = list(fused3_base)

        if bool(RERANK3_MONOT5_PASSAGES):
            assert monot5p_tokenizer is not None
            assert monot5p_model is not None
            assert true_id is not None
            assert false_id is not None

            score_top_n = int(MONOT5P_SCORE_TOP_N) if MONOT5P_SCORE_TOP_N is not None else int(MONOT5P_TOP_N)
            score_top_n = max(int(score_top_n), int(MONOT5P_TOP_N))

            score_max_passages = int(MONOT5P_SCORE_MAX_PASSAGES) if MONOT5P_SCORE_MAX_PASSAGES is not None else int(MONOT5P_MAX_PASSAGES)
            score_max_passages = max(int(score_max_passages), int(MONOT5P_MAX_PASSAGES))

            score_pairs = fused3_for_rerank[: int(score_top_n)]
            score_docids = [d for d, _ in score_pairs]

            raw_key = {
                "qid": str(qid),
                "query": str(query),
                "docids": list(score_docids),
                "model_name": str(MONOT5P_MODEL),
                "device": str(DEVICE),
                #"batch_size": int(MONOT5P_BATCH_SIZE),
                "max_length": int(MONOT5P_MAX_LENGTH),
                "use_fp16": bool(MONOT5P_FP16),
                "doc_max_chars": int(MONOT5P_DOC_MAX_CHARS),
                "passage_chars": int(MONOT5P_PASSAGE_CHARS),
                "stride_chars": int(MONOT5P_STRIDE_CHARS),
                "max_passages": int(score_max_passages),
            }

            raw_scores = disk_cache.get("monot5p_raw", raw_key)
            if raw_scores is None:
                score_texts = fetch_doc_texts_disk_cached(
                    rm3,
                    score_docids,
                    mem_cache=_doc_text_cache,
                    max_chars=int(MONOT5P_DOC_MAX_CHARS),
                    disk_cache=disk_cache,
                )
                raw_scores = compute_monot5_passage_raw_scores(
                    monot5p_tokenizer,
                    monot5p_model,
                    int(true_id),
                    int(false_id),
                    query=str(query),
                    docids=score_docids,
                    doc_texts=score_texts,
                    device=str(DEVICE),
                    batch_size=int(MONOT5P_BATCH_SIZE),
                    max_length=int(MONOT5P_MAX_LENGTH),
                    passage_chars=int(MONOT5P_PASSAGE_CHARS),
                    stride_chars=int(MONOT5P_STRIDE_CHARS),
                    max_passages=int(score_max_passages),
                )
                disk_cache.set("monot5p_raw", raw_key, raw_scores)

            top_pairs = fused3_for_rerank[: int(MONOT5P_TOP_N)]
            top_docids = [d for d, _ in top_pairs]

            extra_top = aggregate_monot5_passage_scores(
                raw_scores,
                top_docids,
                agg=str(MONOT5P_AGG),
                avg_topk=int(MONOT5P_AVG_TOPK),
                max_passages=int(MONOT5P_MAX_PASSAGES),
                softmax_temp=float(MONOT5P_SOFTMAX_TEMP),
                hybrid_lambda=float(MONOT5P_HYBRID_LAMBDA),
            )

            base_scores = {d: float(s) for d, s in top_pairs}
            base_norm = minmax_norm(base_scores)
            extra_norm = minmax_norm({d: float(extra_top.get(d, 0.0)) for d in top_docids})

            alpha = float(MONOT5P_ALPHA)
            comb = {d: alpha * base_norm.get(d, 0.0) + (1.0 - alpha) * extra_norm.get(d, 0.0) for d in top_docids}
            reranked_top = sorted(comb.items(), key=lambda x: (-x[1], x[0]))

            reranked_set = {d for d, _ in reranked_top}
            tail_docids = [d for d, _ in fused3_for_rerank if d not in reranked_set]

            tail_start = (reranked_top[-1][1] if reranked_top else 0.0) - 1.0
            tail_step = 1e-3
            tail_scores = {d: float(tail_start) - float(tail_step) * i for i, d in enumerate(tail_docids, start=1)}

            fused3_final = [(d, float(comb[d])) for d, _ in reranked_top] + [(d, float(tail_scores[d])) for d in tail_docids]
            fused3_final = fused3_final[: int(K)]
        else:
            fused3_final = fused3_for_rerank[: int(K)]

        run_3[str(qid)] = fused3_final

        if i % 10 == 0:
            print(f"processed {i}/{len(target_qids)} queries")

    return run_1, run_2, run_3


def generate_run_2_only() -> Dict[str, List[Tuple[str, float]]]:
    run_2: Dict[str, List[Tuple[str, float]]] = {}

    for i, qid in enumerate(target_qids, start=1):
        query = all_queries[qid]

        q_rm3 = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_RM3),
            hyde_docs=hyde_docs_all,
        )
        q_pp = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_SPLADEPP),
            hyde_docs=hyde_docs_all,
        )
        q_dense = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_DENSE),
            hyde_docs=hyde_docs_all,
        )

        run2_key = {
            'qid': str(qid),
            'query': str(query),
            'query_sources': {
                'rm3': str(QUERY_SOURCE_RM3),
                'spladepp': str(QUERY_SOURCE_SPLADEPP),
                'dense': str(QUERY_SOURCE_DENSE),
            },
            'query_texts': {
                'rm3': str(q_rm3),
                'spladepp': str(q_pp),
                'dense': str(q_dense),
            },
            'k': int(K),
            'rm3': {'index': str(RM3_INDEX), 'bm25': [float(BM25_K1), float(BM25_B)], 'rm3': [int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW)]},
            'run2_sparse': {
                'use_sdm_rm3': bool(RUN2_USE_SDM_RM3),
                'sdm': [float(SDM_TERM_WEIGHT), float(SDM_ORDERED_WINDOW_WEIGHT), float(SDM_UNORDERED_WINDOW_WEIGHT)],
            },
            'spladepp_index': str(SPLADEPP_INDEX),
            'dense_index': str(DENSE_INDEX),
            'dense_ef_search': int(DENSE_EF_SEARCH),
            'w_run2': list(W_RUN2),
        }

        cached = disk_cache.get('generate_runs_run2', run2_key)
        if cached is None:
            if bool(RUN2_USE_SDM_RM3):
                sparse_art = retrieve(rm3, q_rm3, k=int(K), query_generator=sdm_query_generator)
            else:
                sparse_art = retrieve(rm3, q_rm3, k=int(K))

            pp_art = retrieve(spladepp, q_pp, k=int(K))
            dense_art = retrieve(dense, q_dense, k=int(K))

            fallback_zero = [(d, 0.0) for d, _ in sparse_art.ranked]

            fused2 = fuse_weighted_minmax(
                [sparse_art.docids_scores, pp_art.docids_scores, dense_art.docids_scores],
                list(W_RUN2),
                depth=int(K),
            )
            fused2 = ensure_k(fused2, fallback_zero, k=int(K))

            disk_cache.set('generate_runs_run2', run2_key, fused2)
        else:
            fused2 = list(cached)[: int(K)]

        run_2[str(qid)] = list(fused2)

        if i % 10 == 0:
            print(f"processed {i}/{len(target_qids)} queries (run_2 only)")

    return run_2


run2_key = {
    'qid_set': str(QID_SET),
    'k': int(K),
    'rm3': {'index': str(RM3_INDEX), 'bm25': [float(BM25_K1), float(BM25_B)], 'rm3': [int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW)]},
    'run2_sparse': {
        'use_sdm_rm3': bool(RUN2_USE_SDM_RM3),
        'sdm': [float(SDM_TERM_WEIGHT), float(SDM_ORDERED_WINDOW_WEIGHT), float(SDM_UNORDERED_WINDOW_WEIGHT)],
    },
    'spladepp_index': str(SPLADEPP_INDEX),
    'dense_index': str(DENSE_INDEX),
    'dense_ef_search': int(DENSE_EF_SEARCH),
    'query_sources': {
        'rm3': str(QUERY_SOURCE_RM3),
        'spladepp': str(QUERY_SOURCE_SPLADEPP),
        'dense': str(QUERY_SOURCE_DENSE),
    },
    'w_run2': list(W_RUN2),
}

need_run_1 = bool(FORCE_REGEN_RUNS) or (not OUT_RUN_1.exists())
need_run_3 = bool(FORCE_REGEN_RUNS) or (not OUT_RUN_3.exists())
need_run_2 = bool(FORCE_REGEN_RUNS) or (not OUT_RUN_2.exists()) or (not disk_cache.get('written_runs', {'run': 'run_2', **run2_key}))

if (not need_run_1) and (not need_run_2) and (not need_run_3):
    print('All run files already exist for this configuration; nothing to do.')
elif (not need_run_1) and need_run_2 and (not need_run_3):
    t0 = time.time()
    run_2 = generate_run_2_only()
    print('Run_2 generation done in', round(time.time() - t0, 1), 'sec')

    write_trec_run(OUT_RUN_2, run_2, tag='run_2')
    disk_cache.set('written_runs', {'run': 'run_2', **run2_key}, True)
    print('Wrote:', OUT_RUN_2)
else:
    t0 = time.time()
    run_1, run_2, run_3 = generate_all_runs()
    print('Run generation done in', round(time.time() - t0, 1), 'sec')

    if need_run_1:
        write_trec_run(OUT_RUN_1, run_1, tag='run_1')
    if need_run_2:
        write_trec_run(OUT_RUN_2, run_2, tag='run_2')
        disk_cache.set('written_runs', {'run': 'run_2', **run2_key}, True)
    if need_run_3:
        write_trec_run(OUT_RUN_3, run_3, tag='run_3')

    print('Wrote:', OUT_RUN_1, OUT_RUN_2, OUT_RUN_3)

processed 10/199 queries
processed 20/199 queries
processed 30/199 queries


KeyboardInterrupt: 

In [ ]:
# Check requirements and generate runs
# Logic matches the repo's caching and conditional regeneration to save time

run2_key = {
    'qid_set': str(QID_SET),
    'k': int(K),
    'rm3': {'index': str(RM3_INDEX), 'bm25': [float(BM25_K1), float(BM25_B)], 'rm3': [int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW)]},
    'run2_sparse': {
        'use_sdm_rm3': bool(RUN2_USE_SDM_RM3),
        'sdm': [float(SDM_TERM_WEIGHT), float(SDM_ORDERED_WINDOW_WEIGHT), float(SDM_UNORDERED_WINDOW_WEIGHT)],
    },
    'spladepp_index': str(SPLADEPP_INDEX),
    'dense_index': str(DENSE_INDEX),
    'dense_ef_search': int(DENSE_EF_SEARCH),
    'query_sources': {
        'rm3': str(QUERY_SOURCE_RM3),
        'spladepp': str(QUERY_SOURCE_SPLADEPP),
        'dense': str(QUERY_SOURCE_DENSE),
    },
    'w_run2': list(W_RUN2),
}

need_run_1 = bool(FORCE_REGEN_RUNS) or (not OUT_RUN_1.exists())
need_run_3 = bool(FORCE_REGEN_RUNS) or (not OUT_RUN_3.exists())
need_run_2 = bool(FORCE_REGEN_RUNS) or (not OUT_RUN_2.exists()) or (not disk_cache.get('written_runs', {'run': 'run_2', **run2_key}))

if (not need_run_1) and (not need_run_2) and (not need_run_3):
    print('All run files are present and up-to-date.')
elif (not need_run_1) and need_run_2 and (not need_run_3):
    print('Regenerating Run 2 (Structured Queries)...')
    t0 = time.time()
    run_2 = generate_run_2_only()
    write_trec_run(OUT_RUN_2, run_2, tag='run_2')
    disk_cache.set('written_runs', {'run': 'run_2', **run2_key}, True)
    print(f'Done in {round(time.time() - t0, 1)} sec.')
else:
    print('Regenerating all runs...')
    t0 = time.time()
    run_1, run_2, run_3 = generate_all_runs()
    
    if need_run_1:
        write_trec_run(OUT_RUN_1, run_1, tag='run_1')
    if need_run_2:
        write_trec_run(OUT_RUN_2, run_2, tag='run_2')
        disk_cache.set('written_runs', {'run': 'run_2', **run2_key}, True)
    if need_run_3:
        write_trec_run(OUT_RUN_3, run_3, tag='run_3')
        
    print(f'Done in {round(time.time() - t0, 1)} sec.')

Regenerating all runs...
processed 10/199 queries
processed 20/199 queries
processed 30/199 queries
processed 40/199 queries
processed 50/199 queries
processed 60/199 queries


processed 70/199 queries
processed 80/199 queries
processed 90/199 queries
processed 100/199 queries
processed 110/199 queries
processed 120/199 queries
processed 130/199 queries
processed 140/199 queries
processed 150/199 queries
processed 160/199 queries
processed 170/199 queries
processed 180/199 queries
processed 190/199 queries
Done in 1.0 sec.


## 8. Validation: Run File Format

Ensures the generated files meet TREC submission standards:
- 6-column format (`qid Q0 docid rank score tag`)
- Exactly 1000 documents per query
- Correct query IDs for the test set

In [ ]:
def parse_trec_run_lines(path: Path) -> Dict[str, List[Tuple[int, str, float, str]]]:
    run: Dict[str, List[Tuple[int, str, float, str]]] = defaultdict(list)
    with path.open('r', encoding='utf-8') as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 6:
                raise ValueError(f"{path.name}: line {ln} expected 6 columns, got {len(parts)}")
            qid, q0, docid, rank_s, score_s, tag = parts
            if q0 != 'Q0':
                raise ValueError(f"{path.name}: line {ln} col2 expected Q0, got {q0}")
            try:
                rank = int(rank_s)
            except Exception:
                raise ValueError(f"{path.name}: line {ln} rank not int: {rank_s}")
            try:
                score = float(score_s)
            except Exception:
                raise ValueError(f"{path.name}: line {ln} score not float: {score_s}")
            run[str(qid)].append((int(rank), str(docid), float(score), str(tag)))
    return dict(run)


def verify_run_file(path: Path, *, expected_qids: List[str], k: int, expected_tag: Optional[str]) -> None:
    exp = [str(x) for x in expected_qids]
    lines = parse_trec_run_lines(path)

    got = sorted(lines.keys(), key=int)
    if sorted(exp, key=int) != got:
        missing = sorted(set(exp) - set(got), key=int)
        extra = sorted(set(got) - set(exp), key=int)
        raise AssertionError(f"{path.name}: qid mismatch. missing={missing[:10]} extra={extra[:10]}")

    for qid in exp:
        rows = lines[qid]
        if len(rows) != int(k):
            raise AssertionError(f"{path.name}: qid {qid} has {len(rows)} lines, expected {k}")

        ranks = [r for r, _, _, _ in rows]
        if len(set(ranks)) != len(ranks):
            raise AssertionError(f"{path.name}: qid {qid} has duplicate ranks")
        if set(ranks) != set(range(1, int(k) + 1)):
            raise AssertionError(f"{path.name}: qid {qid} ranks are not exactly 1..{k}")

        docids = [d for _, d, _, _ in rows]
        if len(set(docids)) != len(docids):
            raise AssertionError(f"{path.name}: qid {qid} has duplicate docids")

        if expected_tag is not None:
            tags = {t for _, _, _, t in rows}
            if tags != {str(expected_tag)}:
                raise AssertionError(f"{path.name}: qid {qid} has unexpected tags: {sorted(tags)}")


expected_qids = target_qids

print('Verifying:', OUT_RUN_1)
verify_run_file(OUT_RUN_1, expected_qids=expected_qids, k=K, expected_tag='run_1')
print('OK:', OUT_RUN_1.name)

print('Verifying:', OUT_RUN_2)
verify_run_file(OUT_RUN_2, expected_qids=expected_qids, k=K, expected_tag='run_2')
print('OK:', OUT_RUN_2.name)

print('Verifying:', OUT_RUN_3)
verify_run_file(OUT_RUN_3, expected_qids=expected_qids, k=K, expected_tag='run_3')
print('OK:', OUT_RUN_3.name)

Verifying: /root/textretfinal/run_1.res
OK: run_1.res
Verifying: /root/textretfinal/run_2.res
OK: run_2.res
Verifying: /root/textretfinal/run_3.res
OK: run_3.res


## 9. Final Results: Judged MAP

Performance on the 50 judged queries (Validation Set).

In [ ]:
import pandas as pd
from IPython.display import display, HTML

judged_map_key = {
    'qrels_path': str(QRELS_JUDGED_PATH),
    'rm3': {
        'index': str(RM3_INDEX),
        'bm25': [float(BM25_K1), float(BM25_B)],
        'rm3': [int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW)],
    },
    'run2_sparse': {
        'use_sdm_rm3': bool(RUN2_USE_SDM_RM3),
        'sdm': [float(SDM_TERM_WEIGHT), float(SDM_ORDERED_WINDOW_WEIGHT), float(SDM_UNORDERED_WINDOW_WEIGHT)],
    },
    'spladepp_index': str(SPLADEPP_INDEX),
    'spladev3_index': str(SPLADEV3_INDEX),
    'dense_index': str(DENSE_INDEX),
    'dense_query_source': str(QUERY_SOURCE_DENSE),
    'w_run2': list(W_RUN2),
    'w_run3': list(W_RUN3),
    'monot5p': {
        'enabled': bool(RERANK3_MONOT5_PASSAGES),
        'model': str(MONOT5P_MODEL),
        'top_n': int(MONOT5P_TOP_N),
        'alpha': float(MONOT5P_ALPHA),
        #'batch_size': int(MONOT5P_BATCH_SIZE),
        'max_length': int(MONOT5P_MAX_LENGTH),
        'doc_max_chars': int(MONOT5P_DOC_MAX_CHARS),
        'passage_chars': int(MONOT5P_PASSAGE_CHARS),
        'stride_chars': int(MONOT5P_STRIDE_CHARS),
        'max_passages': int(MONOT5P_MAX_PASSAGES),
        'agg': str(MONOT5P_AGG),
        'avg_topk': int(MONOT5P_AVG_TOPK),
        'softmax_temp': float(MONOT5P_SOFTMAX_TEMP),
        'hybrid_lambda': float(MONOT5P_HYBRID_LAMBDA),
        'fp16': bool(MONOT5P_FP16),
    },
}

cached_maps = disk_cache.get('judged_map', judged_map_key)
if cached_maps is None:
    maps = evaluate_judged_maps()
    disk_cache.set('judged_map', judged_map_key, maps)
else:
    maps = cached_maps

results_data = [
    {"Run": "Run 1", "Method": "Baseline Fusion", "MAP": maps['run_1']},
    {"Run": "Run 2", "Method": "Structured Queries (SDM)", "MAP": maps['run_2']},
    {"Run": "Run 3", "Method": "Neural Reranking (MonoT5)", "MAP": maps['run_3']},
]

df = pd.DataFrame(results_data)
display(HTML("<h3>Experimental Results (Judged Queries)</h3>"))
display(df.style.format({"MAP": "{:.4f}"}).hide(axis='index').set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left')]},
    {'selector': 'td', 'props': [('text-align', 'left')]}
]))

## References

- **[ROBUST04 / TREC Robust Track]**
  - Voorhees, E. M. (2004). *Overview of the TREC 2004 Robust Retrieval Track.*
- **[BM25 / Okapi]**
  - Robertson, S., & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.*
- **[Sequential Dependence Model (SDM)]**
  - Metzler, D., & Croft, W. B. (2005). *A Markov Random Field Model for Term Dependencies.*
- **[RM3 pseudo-relevance feedback]**
  - Lavrenko, V., & Croft, W. B. (2001). *Relevance-Based Language Models.* (RM-style feedback foundations)
- **[SPLADE (learned sparse retrieval)]**
  - Formal, T. et al. (2021). *SPLADE: Sparse Lexical and Expansion Model for First Stage Ranking.*
- **[HyDE (Hypothetical Document Embeddings)]**
  - Gao, L. et al. (2023). *Precise Zero-Shot Dense Retrieval without Relevance Labels.*
- **[BGE dense embeddings]**
  - BAAI (2023). *BGE embedding models* (used via Pyserini prebuilt dense index)
- **[MonoT5 reranking]**
  - Nogueira, R. et al. (2020). *Document Ranking with a Pretrained Sequence-to-Sequence Model.*
- **[Pyserini]**
  - Lin, J. et al. (2021). *Pyserini: A Python Toolkit for Reproducible Information Retrieval Research with Sparse and Dense Representations.*

## Reproducibility checklist

- **[Inputs]**
  - Queries: `Files-20260104/queriesROBUST.txt`
  - Judged qrels: `Files-20260104/qrels_50_Queries`
- **[Prebuilt indices]**
  - `robust04`
  - `beir-v1.0.0-robust04.splade-pp-ed`
  - `beir-v1.0.0-robust04.splade-v3`
  - `beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw`
- **[Caching]**
  - Cache directory: `CACHE_DIR` (default `/workspace/.cache`)
- **[Strict rerun switches]**
  - Set `FORCE_REGEN_RUNS=True`
  - Set `FORCE_CACHE_REFRESH=True`
  - Optionally set `FORCE_REGEN_HYDE=True`
